# 01 - Data Preprocessing, Descriptive Statistics & EDA
**Veltris Vehicle Shipment Cost dataset** (`Veltris-Vehicle.xlsx`, 305,838 rows x 145 columns)

This notebook:
1. Loads the raw shipment dataset
2. Computes descriptive statistics on the **entire raw dataset**
3. Cleans the data: drops unusable columns (>50% missing, leakage, high-cardinality IDs), removes duplicate rows, removes rows with missing values, removes outliers (IQR rule) on the target and key continuous drivers
4. Computes descriptive statistics on the **processed dataset**
5. Runs EDA / visualization of feature relationships with the target, organized by business feature group

> The raw `.xlsx` is ~106 MB / 305,838 rows, so it is streamed once to `raw_data.csv` with a fast XML parser (see `src/xml_to_csv.py`) rather than loaded via pandas' default Excel engine.

## 1. Load raw data and convert Excel-serial dates

In [ ]:
"""
Step 1: Load raw data, compute descriptive statistics on the RAW dataset,
clean it (drop near-empty / leakage / high-cardinality-ID columns, drop
duplicate rows, drop rows with missing values, remove outliers on key
continuous columns via IQR), then compute descriptive statistics on the
PROCESSED dataset. Saves both stats tables and the cleaned dataframe.
"""
import pandas as pd
import numpy as np
import json
import time

t0 = time.time()
RAW_CSV = "/home/claude/raw_data.csv"
OUT_DIR = "/home/claude/proj/data/processed"

df = pd.read_csv(RAW_CSV, low_memory=False)
print("Raw shape:", df.shape)

# ---------------------------------------------------------------------
# 0. Convert Excel-serial date columns to real datetimes
# ---------------------------------------------------------------------
DATE_COLS = ["CreationDate", "FirstPickup", "LastPickup",
             "FirstScheduledDelivery", "LastScheduledDelivery",
             "FirstDelivery", "LastDelivery"]
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], unit="D", origin="1899-12-30", errors="coerce")

# ---------------------------------------------------------------------
# 1. Descriptive statistics on the ENTIRE (raw) dataset
# ---------------------------------------------------------------------
num_cols_raw = df.select_dtypes(include=[np.number]).columns.tolist()
desc_raw_numeric = df[num_cols_raw].describe().T
desc_raw_numeric["missing_pct"] = df[num_cols_raw].isna().mean().values * 100
desc_raw_numeric["skew"] = df[num_cols_raw].skew().values
desc_raw_numeric["kurtosis"] = df[num_cols_raw].kurtosis().values

cat_cols_raw = df.select_dtypes(include=["object"]).columns.tolist()
desc_raw_cat = pd.DataFrame({
    "n_unique": df[cat_cols_raw].nunique(),
    "missing_pct": df[cat_cols_raw].isna().mean() * 100,
    "top_value": df[cat_cols_raw].mode().iloc[0] if len(cat_cols_raw) else None,
})

desc_raw_numeric.to_csv(f"{OUT_DIR}/descriptive_stats_RAW_numeric.csv")
desc_raw_cat.to_csv(f"{OUT_DIR}/descriptive_stats_RAW_categorical.csv")
print("Raw descriptive stats saved.")

# ---------------------------------------------------------------------
# 2. Drop columns that are not usable features
#    (near-empty >50% missing, direct target leakage, raw high-card IDs,
#     absolute calendar dates already represented by engineered features,
#     the vendor-provided Split column since we build our own split)
# ---------------------------------------------------------------------
DROP_NEAR_EMPTY = ["InoperableAny", "NetWidth", "OriginTimeZone", "DestinationTimeZone"]
DROP_LEAKAGE = ["TotalCostLog"]  # direct transform of the target
DROP_HIGH_CARD_ID = ["OriginPostalCode", "DestinationPostalCode", "OriginCity", "DestinationCity"]
DROP_RAW_DATES = ["FirstPickup", "LastPickup", "FirstScheduledDelivery",
                   "LastScheduledDelivery", "FirstDelivery", "LastDelivery", "CreationDate"]
DROP_META = ["Split", "ShipmentId"]

drop_cols = DROP_NEAR_EMPTY + DROP_LEAKAGE + DROP_HIGH_CARD_ID + DROP_RAW_DATES + DROP_META
drop_cols = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} unusable columns:", drop_cols)

# ---------------------------------------------------------------------
# 3. Remove duplicate rows
# ---------------------------------------------------------------------
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Duplicate rows removed: {n_before - len(df_clean)}")

# ---------------------------------------------------------------------
# 4. Remove rows with missing values (in the retained feature columns)
# ---------------------------------------------------------------------
n_before = len(df_clean)
df_clean = df_clean.dropna(axis=0, how="any")
print(f"Rows dropped for missing values: {n_before - len(df_clean)} "
      f"({(n_before - len(df_clean)) / n_before * 100:.1f}%)")

# ---------------------------------------------------------------------
# 5. Outlier removal (IQR rule, 1.5x) on target + key continuous drivers
# ---------------------------------------------------------------------
OUTLIER_COLS = ["TotalCost", "TotalMiles", "TotalWeight", "HaversineMiles"]
outlier_report = {}
n_before = len(df_clean)
mask = pd.Series(True, index=df_clean.index)
for c in OUTLIER_COLS:
    q1, q3 = df_clean[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col_mask = df_clean[c].between(lo, hi)
    outlier_report[c] = {
        "lower_bound": float(lo), "upper_bound": float(hi),
        "n_outliers_removed": int((~col_mask).sum())
    }
    mask &= col_mask
df_clean = df_clean[mask].reset_index(drop=True)
print(f"Outlier rows removed (IQR, target+key drivers): {n_before - len(df_clean)} "
      f"({(n_before - len(df_clean)) / n_before * 100:.1f}%)")
print(json.dumps(outlier_report, indent=2))

with open(f"{OUT_DIR}/outlier_report.json", "w") as f:
    json.dump(outlier_report, f, indent=2)

print("Processed shape:", df_clean.shape)

# ---------------------------------------------------------------------
# 6. Descriptive statistics on the PROCESSED dataset
# ---------------------------------------------------------------------
num_cols_p = df_clean.select_dtypes(include=[np.number]).columns.tolist()
desc_p_numeric = df_clean[num_cols_p].describe().T
desc_p_numeric["missing_pct"] = df_clean[num_cols_p].isna().mean().values * 100
desc_p_numeric["skew"] = df_clean[num_cols_p].skew().values
desc_p_numeric["kurtosis"] = df_clean[num_cols_p].kurtosis().values

cat_cols_p = df_clean.select_dtypes(include=["object"]).columns.tolist()
desc_p_cat = pd.DataFrame({
    "n_unique": df_clean[cat_cols_p].nunique(),
    "missing_pct": df_clean[cat_cols_p].isna().mean() * 100,
    "top_value": df_clean[cat_cols_p].mode().iloc[0] if len(cat_cols_p) else None,
})

desc_p_numeric.to_csv(f"{OUT_DIR}/descriptive_stats_PROCESSED_numeric.csv")
desc_p_cat.to_csv(f"{OUT_DIR}/descriptive_stats_PROCESSED_categorical.csv")

# ---------------------------------------------------------------------
# 7. Persist
# ---------------------------------------------------------------------
df_clean.to_parquet(f"{OUT_DIR}/veltris_cleaned.parquet", index=False)

summary = {
    "raw_shape": list(df.shape),
    "processed_shape": list(df_clean.shape),
    "columns_dropped_unusable": drop_cols,
    "rows_dropped_duplicates": int(n_before) - int(n_before),  # placeholder overwritten below
}
summary["outlier_report"] = outlier_report
with open(f"{OUT_DIR}/preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("DONE preprocessing in", round(time.time() - t0, 1), "s")


**Result of running the cell above:**
```
Raw shape: (305838, 145)
Raw descriptive stats saved.
Dropped 18 unusable columns: ['InoperableAny', 'NetWidth', 'OriginTimeZone', 'DestinationTimeZone', 'TotalCostLog', 'OriginPostalCode', 'DestinationPostalCode', 'OriginCity', 'DestinationCity', 'FirstPickup', 'LastPickup', 'FirstScheduledDelivery', 'LastScheduledDelivery', 'FirstDelivery', 'LastDelivery', 'CreationDate', 'Split', 'ShipmentId']
Duplicate rows removed: 115
Rows dropped for missing values: 133030 (43.5%)
Outlier rows removed (IQR, target+key drivers): 25072 (14.5%)
Processed shape: (147621, 127)
DONE preprocessing in 17.1 s
```

### Why these columns were dropped
- **Near-empty (>50% missing):** `InoperableAny` (98.6%), `NetWidth` (95.0%), `OriginTimeZone` (80.2%), `DestinationTimeZone` (79.5%) - too sparse to impute reliably.
- **Direct target leakage:** `TotalCostLog` is `log(TotalCost)`, a deterministic transform of the label.
- **High-cardinality raw identifiers:** `OriginPostalCode`/`DestinationPostalCode` (5-digit zip), `OriginCity`/`DestinationCity` - superseded by `OriginZip3`/`DestinationZip3` and the target-encoded `OriginZip3_TE`/`DestinationZip3_TE`.
- **Raw absolute calendar dates:** superseded by the already-engineered cyclical/seasonal features (`CreationDate_day_of_year_sin/cos`, `is_weekend`, `is_eoq`, `is_holiday`, `lead_time_days`, `delivery_date_days`); `FirstDelivery`/`LastDelivery` additionally are post-outcome timestamps and would leak information not available at pricing time.
- **`Split`**: a pre-existing train/val/test tag from the source system - we build our own 80/10/10 split per the project brief. **`ShipmentId`**: a unique row identifier, not a feature.

### Missing-value / outlier strategy
Rows with missing values in the *retained* columns were dropped (43.5% of rows) rather than imputed, per the cleaning brief. Outliers were removed with the 1.5xIQR rule on the target (`TotalCost`) and the three most influential continuous drivers (`TotalMiles`, `TotalWeight`, `HaversineMiles`).

**Important finding:** row-wise missing-value deletion removes the **entire Magnus segment** (`EquipmentType`/`Enclosed` are 100% missing for Magnus shipments). This is handled explicitly in notebook `03_segment_datasets.ipynb`, where Magnus rows are imputed instead of dropped.

## 2. Descriptive statistics: RAW vs PROCESSED dataset
Full tables are saved to `data/processed/descriptive_stats_RAW_numeric.csv`, `descriptive_stats_RAW_categorical.csv`, `descriptive_stats_PROCESSED_numeric.csv`, `descriptive_stats_PROCESSED_categorical.csv`.

In [ ]:
import pandas as pd
raw_num = pd.read_csv('../data/processed/descriptive_stats_RAW_numeric.csv', index_col=0)
proc_num = pd.read_csv('../data/processed/descriptive_stats_PROCESSED_numeric.csv', index_col=0)
print('RAW numeric summary (TotalCost row):')
print(raw_num.loc['TotalCost'])
print('\nPROCESSED numeric summary (TotalCost row):')
print(proc_num.loc['TotalCost'])


## 3. EDA & visualization - feature groups vs target

In [1]:
"""
Step 3: EDA & visualization - relationship of the (VIF+Boruta) reduced
feature set with the target (TotalCost), organized by business feature group.
Saves PNG charts to artifacts/figures.
"""
import sys
sys.path.insert(0, "/home/claude/proj/src")
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from feature_groups import FEATURE_GROUPS

sns.set_style("whitegrid")
FIG_DIR = "/home/claude/proj/artifacts/figures"
DATA_DIR = "/home/claude/proj/data/processed"
TARGET = "TotalCost"

df = pd.read_parquet(f"{DATA_DIR}/veltris_reduced.parquet")
final_features = json.load(open(f"{DATA_DIR}/feature_selection_summary.json"))["final_features"]
print("Reduced data:", df.shape, "| features:", len(final_features))

# map each final feature to its group
feat_to_group = {}
for g, feats in FEATURE_GROUPS.items():
    for f in feats:
        feat_to_group[f] = g
group_of_feature = {f: feat_to_group.get(f, "Other") for f in final_features}
pd.Series(group_of_feature, name="group").to_csv(f"{DATA_DIR}/final_feature_group_mapping.csv")
print(pd.Series(group_of_feature).value_counts())

# ---------------------------------------------------------------------
# 1. Target distribution (raw dataset vs processed/outlier-removed)
# ---------------------------------------------------------------------
raw = pd.read_csv("/home/claude/raw_data.csv", low_memory=False, usecols=[TARGET])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(raw[TARGET].dropna(), bins=80, color="#4C72B0")
axes[0].set_title("TotalCost distribution - RAW dataset")
axes[0].set_xlabel("TotalCost ($)")
axes[1].hist(df[TARGET], bins=80, color="#55A868")
axes[1].set_title("TotalCost distribution - PROCESSED (outliers removed)")
axes[1].set_xlabel("TotalCost ($)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_target_distribution_before_after.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 2. Correlation heatmap of final reduced features + target
# ---------------------------------------------------------------------
num_final = [f for f in final_features if df[f].dtype != "object"]
corr = df[num_final + [TARGET]].corr()
plt.figure(figsize=(13, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True,
            cbar_kws={"shrink": 0.7})
plt.title("Correlation heatmap - reduced feature set vs TotalCost")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_correlation_heatmap_reduced.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 3. Correlation of each reduced feature with target, colored by group
# ---------------------------------------------------------------------
target_corr = corr[TARGET].drop(TARGET).sort_values()
colors = sns.color_palette("tab10", n_colors=len(FEATURE_GROUPS))
group_color_map = {g: colors[i] for i, g in enumerate(FEATURE_GROUPS.keys())}
bar_colors = [group_color_map.get(group_of_feature.get(f, "Other"), "grey") for f in target_corr.index]

plt.figure(figsize=(9, 8))
plt.barh(target_corr.index, target_corr.values, color=bar_colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation of each selected feature with TotalCost\n(color = feature group)")
plt.xlabel("Pearson correlation with TotalCost")
handles = [plt.Rectangle((0, 0), 1, 1, color=group_color_map[g]) for g in FEATURE_GROUPS]
plt.legend(handles, FEATURE_GROUPS.keys(), bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_feature_target_correlation_by_group.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 4. Scatter/relationship plots for top 4 correlated features
# ---------------------------------------------------------------------
top_feats = target_corr.abs().sort_values(ascending=False).head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, f in zip(axes.ravel(), top_feats):
    sample = df.sample(min(15000, len(df)), random_state=1)
    ax.hexbin(sample[f], sample[TARGET], gridsize=40, cmap="Blues", mincnt=1)
    ax.set_xlabel(f)
    ax.set_ylabel("TotalCost")
    ax.set_title(f"{f} vs TotalCost (r={corr.loc[f, TARGET]:.2f})")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_top_feature_relationships.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 5. Distance & lane-cost group relationship (business-relevant view)
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sample = df.sample(min(20000, len(df)), random_state=1)
axes[0].hexbin(sample["HaversineMiles"], sample[TARGET], gridsize=40, cmap="Oranges", mincnt=1)
axes[0].set_title("Distance group: HaversineMiles vs TotalCost")
axes[0].set_xlabel("HaversineMiles"); axes[0].set_ylabel("TotalCost")
axes[1].hexbin(sample["TotalCostOriginMean"], sample[TARGET], gridsize=40, cmap="Greens", mincnt=1)
axes[1].set_title("Historical Lane Cost group: TotalCostOriginMean vs TotalCost")
axes[1].set_xlabel("TotalCostOriginMean"); axes[1].set_ylabel("TotalCost")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_distance_lanecost_relationships.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 6. Boxplot: TotalCost by EquipmentType (Equipment group) & OriginState top10 (Customer group)
# ---------------------------------------------------------------------
df_full_cleaned = pd.read_parquet(f"{DATA_DIR}/veltris_cleaned.parquet")
fig, axes = plt.subplots(1, 1, figsize=(9, 5))
order = df_full_cleaned.groupby("EquipmentType")[TARGET].median().sort_values().index
sns.boxplot(data=df_full_cleaned, x="EquipmentType", y=TARGET, order=order, ax=axes, showfliers=False)
axes.set_title("Equipment group: TotalCost by EquipmentType")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/06_equipment_type_cost.png", dpi=130)
plt.close()

# ---------------------------------------------------------------------
# 7. Group-level "average |correlation| with target" summary bar
# ---------------------------------------------------------------------
group_scores = {}
for g, feats in FEATURE_GROUPS.items():
    present = [f for f in feats if f in num_final]
    if present:
        group_scores[g] = corr.loc[present, TARGET].abs().mean()
if group_scores:
    gs = pd.Series(group_scores).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    gs.plot(kind="bar", color=[group_color_map[g] for g in gs.index])
    plt.title("Average |correlation| with TotalCost by feature group\n(selected features only)")
    plt.ylabel("Mean |Pearson r|")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/07_group_level_avg_correlation.png", dpi=130)
    plt.close()

print("EDA figures saved to", FIG_DIR)


Reduced data: (147621, 26) | features: 24
Historical Lane Cost    12
Distance                 5
Equipment                4
Market Conditions        3
EDA figures saved to ../artifacts/figures

Figures produced (saved under `artifacts/figures/`):
1. `01_target_distribution_before_after.png` - TotalCost distribution, raw vs outlier-removed
2. `02_correlation_heatmap_reduced.png` - correlation heatmap of the reduced feature set
3. `03_feature_target_correlation_by_group.png` - each selected feature's correlation with TotalCost, colored by business feature group
4. `04_top_feature_relationships.png` - hexbin plots for the 4 most correlated features
5. `05_distance_lanecost_relationships.png` - Distance & Historical-Lane-Cost group views
6. `06_equipment_type_cost.png` - TotalCost by EquipmentType (Equipment group)
7. `07_group_level_avg_correlation.png` - average |correlation| with target per feature group